# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q duckdb huggingface_hub pyarrow pandas

In [3]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
import duckdb

In [4]:
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

In [5]:
con.sql("""
    INSTALL httpfs;
    LOAD httpfs;
    INSTALL parquet;
    LOAD parquet;
""")

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

con.sql(f"""
    CREATE OR REPLACE VIEW fact_content_daily_performance AS
    SELECT *
    FROM read_parquet('{march_path}');
""")

print("March 2026 warehouse data connected.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March 2026 warehouse data connected.


### 1. Unit of analysis + time window

**Unit of analysis:** One row represents the daily performance of one content item for one client on one report date.

**Time window:** I will use March 2026 (`2026-03-01` to `2026-03-31`) as the development window. I am using a mid-panel month rather than the final June 2026 month so that the final month remains a sealed outcome/test window.

In [6]:
con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM fact_content_daily_performance
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5;
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘



In [7]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM fact_content_daily_performance;
""").show()

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

### Features

### Features

I will use five observed performance features:

- `gsc_impressions` — search visibility measured by Google Search Console impressions.
- `gsc_clicks` — clicks received from Google Search.
- `gsc_avg_position` — average Google Search position observed for the content item.
- `sessions_ai` — sessions referred from AI tools.
- `gsc_measured_days` — number of days with usable GSC measurements.

These are observed performance signals available before the prediction decision in this feature-frame design.

### Label / proxy

The task is to identify content that may be declining and prioritize it for review. Because this daily warehouse table does not contain the starter dataset's precomputed decline label, I will construct a clearly defined decline proxy from the observed performance window rather than treating a future outcome as a feature.

### Context

- `report_date`
- `month`
- `client_hash_id`
- `content_hash_id`
- `gsc_data_available`
- `ga4_data_available`

These fields provide time, identity, grouping, and data-availability context. They are not used as model features.

### Excluded

- `client_hash_id` and `content_hash_id` — identifiers used for grouping or joining, not predictive inputs.
- `gsc_data_available` and `ga4_data_available` — availability indicators used to interpret missing data, not performance features.
- Any future-window or label-derived field — excluded because it would leak information from the outcome into the features.
- Any field derived from the decline proxy — excluded from the feature set for the same leakage reason.

### Five-feature frame

I use five features for the first feature frame:

1. **`gsc_impressions`** — knowable at the decision moment because it is observed GSC impression data from the available performance period.
2. **`gsc_clicks`** — knowable at the decision moment because it is observed GSC click data from the available performance period.
3. **`gsc_avg_position`** — knowable at the decision moment because it is calculated from observed GSC position data.
4. **`sessions_ai`** — knowable at the decision moment because AI-referred sessions are an observed traffic signal in the warehouse.
5. **`gsc_measured_days`** — knowable at the decision moment because it records how many days had usable GSC measurements.

The `decline_proxy` is the label/proxy being predicted and is not used as a feature. It represents whether the content-client pair recorded zero GSC clicks in the later outcome window.

In [8]:
feature_frame = con.sql("""
    WITH feature_window AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS gsc_impressions,
            SUM(gsc_clicks) AS gsc_clicks,

            SUM(gsc_sum_position) * 1.0
                / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position,

            SUM(sessions_ai) AS sessions_ai,

            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS gsc_measured_days

        FROM fact_content_daily_performance
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-15'
          AND gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    label_window AS (
        SELECT
            client_hash_id,
            content_hash_id,

            CASE
                WHEN SUM(gsc_clicks) = 0 THEN 1
                ELSE 0
            END AS decline_proxy

        FROM fact_content_daily_performance
        WHERE report_date BETWEEN DATE '2026-03-16'
                              AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        f.sessions_ai,
        f.gsc_measured_days,
        l.decline_proxy

    FROM feature_window f
    INNER JOIN label_window l
        ON f.client_hash_id = l.client_hash_id
       AND f.content_hash_id = l.content_hash_id
""").df()

feature_frame.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_ai,gsc_measured_days,decline_proxy
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,3.964912,NaN,15,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,4974.0,9.0,8.088460,NaN,15,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,35.0,0.0,4.857143,NaN,15,1
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,398.0,1.0,5.298995,NaN,15,1
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,37.0,0.0,17.837838,NaN,13,1


### Deliberate leakage experiment

I deliberately added `decline_proxy` itself as `leak_feature` to demonstrate target leakage. Because the feature directly contains the future outcome, the resulting score becomes perfect.

This result is not evidence of a good model. It demonstrates why a label-derived or future field must never be included among the prediction features.

In [9]:
leaky_frame = feature_frame.copy()

# Deliberately add the future label as if it were a feature
leaky_frame["leak_feature"] = leaky_frame["decline_proxy"]

leaky_accuracy = (
    leaky_frame["leak_feature"] == leaky_frame["decline_proxy"]
).mean()

print(f"Accuracy with deliberate leakage: {leaky_accuracy:.3f}")

Accuracy with deliberate leakage: 1.000


### Honest feature frame after leakage removal

The deliberately leaked `leak_feature` has been removed. The final feature set contains only the five fields defined as knowable at the decision moment.

The `decline_proxy` remains only as the label/proxy and is not included among the prediction features.

The perfect 1.000 result from the leakage experiment is therefore discarded and is not treated as a model performance result.

In [13]:
honest_frame = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_ai",
        "gsc_measured_days",
        "decline_proxy"
    ]
].copy()

print("Leakage feature removed.")
print("Honest feature columns:")
print([
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_ai",
    "gsc_measured_days"
])

Leakage feature removed.
Honest feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'sessions_ai', 'gsc_measured_days']


## 3. Verify it with queries (grain, counts, missing values, windows)
### Verification results

**1. Grain check:** I checked whether more than one row exists for the same `report_date`, `client_hash_id`, and `content_hash_id`. The query returned 0 rows, supporting the stated daily grain.

**2. Time-window check:** The March 2026 slice contains 9,841,378 rows, with dates from 2026-03-01 through 2026-03-31.

**3. GSC availability check:** Using `gsc_data_available IS TRUE`, 3,611,061 rows have usable GSC data in the March slice.

In [12]:
con.sql("""
    SELECT
        COUNT(*) AS rows_with_gsc_data
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE;
""").show()

┌────────────────────┐
│ rows_with_gsc_data │
│       int64        │
├────────────────────┤
│            3611061 │
└────────────────────┘



### 4. Data limits

- GSC data is not available for every row, so rows without usable GSC measurements cannot be interpreted as having zero search activity.
- `sessions_ai` contains 66,245 missing values in the feature frame, so AI-referred session data is not available for every content-client pair.
- The warehouse contains pseudonymized client and content IDs, so they are useful for grouping and joining but should not be treated as predictive features.
- The `decline_proxy` is only a proxy for content decline; it is not the original `trend_direction` label from the starter dataset.
- The feature frame is based on the March 2026 development slice, so results from this exercise should not be treated as evidence of general model performance.

In [14]:
# Check missing values in the five selected features
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_ai",
    "gsc_measured_days"
]

missing_summary = feature_frame[feature_columns].isna().sum()

print("Missing values in selected features:")
print(missing_summary)


Missing values in selected features:
gsc_impressions          0
gsc_clicks               0
gsc_avg_position         0
sessions_ai          66245
gsc_measured_days        0
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.